# 01_dataset_builder - Objetivo y contrato de salida (derivado de JSON)

Este notebook define el contrato de salida del dataset de backtest usando **solo artefactos JSON** de `runs/data_quality`.

Regla metodologica: cada decision debe indicar (a) fuente JSON, (b) politica, (c) respuesta operativa para el dataset builder.

**1) Objetivo formal**

Construir un dataset `ticker-minute` reproducible para backtest, con universo elegible, periodo valido, sesion valida, columnas canonicas y trazabilidad de exclusiones.

Contrato minimo de salida:
- `dataset.parquet`
- `dataset_manifest.json`
- `exclusions.parquet`
- `build_report.json`

Todos los criterios de inclusion/exclusion se derivan de politicas de `03, 04, 05, 06, 09, 10, 11, 12`.

In [1]:
from __future__ import annotations
import json
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path(r"C:/TSIS_Data/v1/backtest_SmallCaps")
DQ = ROOT / "runs" / "data_quality"

CHECK_FILES = {
    "03": ("03_time_coverage", "decisions_03.json"),
    "04": ("04_calendar_definition", "calendar_reconciliation_decision.json"),
    "05": ("05_session_rules", "session_rules_decision.json"),
    "06": ("06_ohlcv_vs_quotes", "decisions_06.json"),
    "09": ("09_sequence_timestamp_integrity", "sequence_timestamp_decision.json"),
    "10": ("10_clock_drift_qa", "clock_drift_decision.json"),
    "11": ("11_nbbo_spread_sanity", "nbbo_spread_decision.json"),
    "12": ("12_off_exchange_trf_reconciliation", "off_exchange_decision.json"),
}

def latest_json(check_dir: str, filename: str) -> Path:
    candidates = sorted((DQ / check_dir).rglob(filename))
    if not candidates:
        raise FileNotFoundError(f"No encontrado: {check_dir}/{filename}")
    return candidates[-1]

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

decision_paths = {}
decisions = {}
for chk, (folder, fname) in CHECK_FILES.items():
    p = latest_json(folder, fname)
    decision_paths[chk] = p
    decisions[chk] = load_json(p)

print("Fuentes JSON activas (latest por check):")
for chk in sorted(decision_paths):
    print(f"- {chk}: {decision_paths[chk]}")
print("\nAs-of UTC:", datetime.now(timezone.utc).isoformat())

Fuentes JSON activas (latest por check):
- 03: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\03_time_coverage\20260216_161633\decisions_03.json
- 04: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\04_calendar_definition\20260212_192637_72ab7c43\calendar_reconciliation_decision.json
- 05: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\05_session_rules\20260211_191206_1cfd88a6\session_rules_decision.json
- 06: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\06_ohlcv_vs_quotes\20260216_170328_51485843\decisions_06.json
- 09: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\09_sequence_timestamp_integrity\20260213_124407_d854fc91\sequence_timestamp_decision.json
- 10: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\10_clock_drift_qa\20260213_181527_74d5d23c\clock_drift_decision.json
- 11: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\11_nbbo_spread_sanity\20260213_182538_4fe3b626\nbbo_spread_decision.json
- 12: C:\TSIS_Data\v1\backtest_SmallCaps\runs

**2) Lectura de politicas y estados**

Esta celda responde al contrato: "mostrar de donde sale cada politica".

Criterio cientifico: una politica es valida para el builder solo si esta en JSON versionado y trazable por ruta de artefacto.

In [2]:
from pprint import pprint

policy_view = {
    "03_status": decisions["03"].get("status"),
    "04_dataset_policy": decisions["04"].get("dataset_policy"),
    "04_overall_status": decisions["04"].get("overall_status"),
    "05_timezone": decisions["05"].get("timezone_canonical"),
    "05_policy": decisions["05"].get("policy"),
    "06_status": decisions["06"].get("status"),
    "06_thresholds": decisions["06"].get("thresholds"),
    "09_overall_status": decisions["09"].get("overall_status"),
    "10_overall_status": decisions["10"].get("overall_status"),
    "11_overall_status": decisions["11"].get("overall_status"),
    "12_overall_status": decisions["12"].get("overall_status"),
}
pprint(policy_view, sort_dicts=False)

{'03_status': {'sample_representativeness': 'FAIL',
               'time_alignment_coherence': 'WARN',
               'overall': 'FAIL'},
 '04_dataset_policy': {'non_gating': ['quotes_p95'],
                       'quotes_p95_min_coverage': 0.25},
 '04_overall_status': 'WARN',
 '05_timezone': 'America/New_York',
 '05_policy': {'premarket_start': '04:00:00',
               'postmarket_end_regular': '20:00:00',
               'postmarket_end_halfday': '17:00:00',
               'signal_allowed': ['premarket', 'rth'],
               'execution_allowed': ['premarket', 'rth']},
 '06_status': {'sample_representativeness': 'FAIL',
               'price_coherence': 'WARN',
               'coherence_components': {'p_quote_range_intersects_ohlcv': 'PASS',
                                        'p_in_range_last_mid': 'WARN',
                                        'd_norm_p95': 'WARN'},
               'overall': 'FAIL'},
 '06_thresholds': {'sample_coverage_pass': 0.95,
                   'sample

**3) Contrato: preguntas obligatorias y respuesta derivada**

Definimos preguntas cerradas para evitar ambiguedad:
1. Universo: que estados de QA entran al dataset operativo.
2. Periodo: que rango temporal es valido segun artefactos.
3. Sesion: que sesiones/minutos son elegibles para senal/ejecucion.
4. Columnas: que columnas minimas y llaves canonicas se exigen.
5. Granularidad: cual es el bucket temporal canonico.
6. Trazabilidad: como justificamos exclusiones y riesgo residual.

In [3]:
def get(obj, *keys, default=None):
    cur = obj
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

status03 = get(decisions, "03", "status", default={})
status06 = get(decisions, "06", "status", default={})
policy05 = get(decisions, "05", "policy", default={})
thr06 = get(decisions, "06", "thresholds", default={})

contract_answers = {
    "universo_operativo": {
        "rule": "Incluir GO y WARN; excluir FAIL y NOT_COMPARABLE del set operable.",
        "evidence": {
            "03.overall": status03.get("overall"),
            "06.overall": status06.get("overall"),
            "10.overall": get(decisions, "10", "overall_status"),
            "11.overall": get(decisions, "11", "overall_status"),
            "12.overall": get(decisions, "12", "overall_status"),
        },
    },
    "periodo_valido": {
        "rule": "Derivar periodo comun por interseccion de disponibilidad y check de cobertura (03).",
        "evidence": {
            "03.metrics_keys": list(get(decisions, "03", "metrics", default={}).keys()),
        },
    },
    "sesion_y_timezone": {
        "rule": "Usar timezone canonica America/New_York y sesiones permitidas por 05.",
        "evidence": {
            "timezone": get(decisions, "05", "timezone_canonical"),
            "policy": policy05,
        },
    },
    "granularidad": {
        "rule": "Minuto canonico para join de mercado, llaves (ticker, minute_ny).",
        "evidence": {
            "06.minute_bucket": get(decisions, "06", "minute_bucket"),
        },
    },
    "comparabilidad_ohlcv_quotes": {
        "rule": "Gate multi-eje (sample coverage + coherence de precio) con umbrales del 06.",
        "evidence": {
            "06.thresholds": thr06,
            "06.coherence_components": get(decisions, "06", "status", "coherence_components", default={}),
        },
    },
    "trazabilidad": {
        "rule": "Cada exclusion debe mapear a check_id + metrica + umbral + estado.",
        "evidence": {
            "sources": {k: str(v) for k, v in decision_paths.items()},
        },
    },
}

from pprint import pprint
pprint(contract_answers, sort_dicts=False)

{'universo_operativo': {'rule': 'Incluir GO y WARN; excluir FAIL y '
                                'NOT_COMPARABLE del set operable.',
                        'evidence': {'03.overall': 'FAIL',
                                     '06.overall': 'FAIL',
                                     '10.overall': 'NOT_APPLICABLE',
                                     '11.overall': 'PASS',
                                     '12.overall': 'NOT_APPLICABLE'}},
 'periodo_valido': {'rule': 'Derivar periodo comun por interseccion de '
                            'disponibilidad y check de cobertura (03).',
                    'evidence': {'03.metrics_keys': ['coverage_mean',
                                                     'violation_ratio_full',
                                                     'truncated_days_exact_50k']}},
 'sesion_y_timezone': {'rule': 'Usar timezone canonica America/New_York y '
                               'sesiones permitidas por 05.',
                       'evidenc

**4) Marco matematico para decision de inclusion**

Definiciones:
- `retention_rate = N_kept / N_candidate`
- `violation_rate = N_violations / N_join_minutes`
- `coverage_gap = 1 - sample_coverage`

Criterio de gate multi-eje (06):
- Eje A (representatividad): `sample_coverage` contra umbrales pass/warn.
- Eje B (coherencia de precio): `p_quote_range_intersects`, `p_in_range`, `d_norm_p95`.

Interpretacion didactica:
- Un `FAIL` en representatividad implica sesgo de muestra alto: no hay soporte estadistico robusto para extrapolar comportamiento de ejecucion.
- Un `WARN` de coherencia sugiere ruido/fragilidad de microestructura: se permite analisis exploratorio, no despliegue operativo ciego.

In [4]:
m03 = get(decisions, "03", "metrics", default={})
m06 = get(decisions, "06", "metrics_global", default={})

# Compatibilidad: algunos runs usan sample_coverage_global y otros sample_coverage
sample_coverage = m06.get("sample_coverage_global", m06.get("sample_coverage"))
if isinstance(sample_coverage, (int, float)):
    coverage_gap = 1.0 - sample_coverage
    print(f"sample_coverage={sample_coverage:.6f}, coverage_gap={coverage_gap:.6f}")
else:
    print("sample_coverage no disponible en metrics_global de 06")

# Compatibilidad: p_in_range puede venir con diferentes nombres segun version
p_in_range = m06.get("p_in_range", m06.get("p_in_range_last_mid"))
p_overlap = m06.get("p_quote_range_intersects_ohlcv")
d_norm_p95 = m06.get("d_norm_p95")

print("")
print("Metricas eje coherencia (06):")
print("- p_quote_range_intersects_ohlcv:", p_overlap)
print("- p_in_range:", p_in_range)
print("- d_norm_p95:", d_norm_p95)

if p_in_range is not None and p_overlap is not None:
    print("")
    print("Lectura practica:")
    print("- usar p_overlap como eje principal de interseccion geometrica (compatibilidad de rango)")
    print("- usar p_in_range como eje secundario de consistencia puntual")


sample_coverage=0.804773, coverage_gap=0.195227

Metricas eje coherencia (06):
- p_quote_range_intersects_ohlcv: 0.9997034400948992
- p_in_range: 0.8072360616844603
- d_norm_p95: 0.2857142836734114

Lectura practica:
- usar p_overlap como eje principal de interseccion geometrica (compatibilidad de rango)
- usar p_in_range como eje secundario de consistencia puntual


**5) Tabla final GO/WARN/FAIL/NOT_COMPARABLE con trazabilidad**

Esta tabla es el corazon del contrato. Debe existir antes de construir el `dataset.parquet` final.

Taxonomia de salida (por ticker):
- `GO`: apto para entrenamiento/evaluacion principal.
- `WARN`: apto con etiqueta de riesgo (reportar separado).
- `FAIL`: excluido del universo operativo.
- `NOT_COMPARABLE`: fuera de comparables (mantener solo para diagnostico).

In [5]:
import pandas as pd

rows = []
rows.append({
    "check": "03_time_coverage",
    "status": get(decisions, "03", "status", "overall"),
    "root_cause": "Cobertura temporal y coherencia de join OHLCV-quotes",
    "source_json": str(decision_paths["03"]),
})
rows.append({
    "check": "04_calendar_definition",
    "status": get(decisions, "04", "overall_status"),
    "root_cause": "Reconciliacion calendario + politica non-gating quotes_p95",
    "source_json": str(decision_paths["04"]),
})
rows.append({
    "check": "05_session_rules",
    "status": "POLICY",
    "root_cause": "Define sesiones y timezone canonica para inclusion temporal",
    "source_json": str(decision_paths["05"]),
})
rows.append({
    "check": "06_ohlcv_vs_quotes",
    "status": get(decisions, "06", "status", "overall"),
    "root_cause": "Comparabilidad de escala y coherencia microestructura",
    "source_json": str(decision_paths["06"]),
})
rows.append({
    "check": "09_sequence_timestamp_integrity",
    "status": get(decisions, "09", "overall_status"),
    "root_cause": "Orden temporal, duplicados y monotonicidad",
    "source_json": str(decision_paths["09"]),
})
rows.append({
    "check": "10_clock_drift_qa",
    "status": get(decisions, "10", "overall_status"),
    "root_cause": "Aplicabilidad de drift de reloj (dual clock)",
    "source_json": str(decision_paths["10"]),
})
rows.append({
    "check": "11_nbbo_spread_sanity",
    "status": get(decisions, "11", "overall_status"),
    "root_cause": "Sanity de spread NBBO y cruces",
    "source_json": str(decision_paths["11"]),
})
rows.append({
    "check": "12_off_exchange_trf_reconciliation",
    "status": get(decisions, "12", "overall_status"),
    "root_cause": "Aplicabilidad de reconciliacion TRF por mapping de venues",
    "source_json": str(decision_paths["12"]),
})

decision_table = pd.DataFrame(rows)
decision_table

,check,status,root_cause,source_json
0,03_time_coverage,FAIL,Cobertura temporal y coherencia de join OHLCV-...,C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
1,04_calendar_definition,WARN,Reconciliacion calendario + politica non-gatin...,C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
2,05_session_rules,POLICY,Define sesiones y timezone canonica para inclu...,C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
3,06_ohlcv_vs_quotes,FAIL,Comparabilidad de escala y coherencia microest...,C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
4,09_sequence_timestamp_integrity,WARN,"Orden temporal, duplicados y monotonicidad",C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
5,10_clock_drift_qa,NOT_APPLICABLE,Aplicabilidad de drift de reloj (dual clock),C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
6,11_nbbo_spread_sanity,PASS,Sanity de spread NBBO y cruces,C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...
7,12_off_exchange_trf_reconciliation,NOT_APPLICABLE,Aplicabilidad de reconciliacion TRF por mappin...,C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_q...


**6) Politica operacional del Dataset Builder (respuesta final)**

Reglas ejecutables:
- Universo: incluir `GO` y `WARN`; excluir `FAIL`; aislar `NOT_COMPARABLE`.
- Periodo: interseccion temporal comun con cobertura minima verificable por 03.
- Sesion: solo minutos dentro de politica 05 (`premarket` + `rth` para senal/ejecucion).
- Granularidad: minuto canonico (`minute_bucket` del 06).
- Coherencia de precio: aplicar gate multi-eje del 06 antes de consolidar comparables.
- QA complementaria: incorporar 09 y 11 en gating; 10 y 12 como `NOT_APPLICABLE` cuando corresponda.

Salida didactica exigida en `build_report.json`:
- Cobertura retenida, tasa de exclusion y causa raiz por ticker.
- Umbral aplicado y valor observado por cada check.
- Riesgo residual explicitado (no solo etiqueta de estado).